# Woche 2 – Tag 2: Tabellen verknüpfen (Joins)

## Ziel
Die bereinigten Bestellungen werden mit dem Kundenstamm und dem
Produktkatalog verknüpft.

Dabei soll sichergestellt werden, dass:
- keine Bestellungen verloren gehen,
- keine zusätzlichen Bestellungen durch doppelte Schlüssel entstehen,
- fehlende Zuordnungen erkannt und dokumentiert werden,
- die Umsatzberechnung korrekt bleibt.

## Join-Verträge – Vorbereitung

Vor der Umsetzung werden die Schlüssel, die erwarteten Beziehungen
und die Qualitätsprüfungen für beide Joins festgelegt.

In [1]:
import pandas as pd
from pathlib import Path

ordner = Path.home() / "Desktop" / "Modul 5" / "Aufgaben" / "Woche_2"

bestellungen = pd.read_parquet(ordner / "bestellungen_q1_clean.parquet")
kunden = pd.read_csv(ordner / "kunden.csv", sep=None, engine="python")
produkte = pd.read_csv(ordner / "produkte.csv", sep=None, engine="python")

print("BESTELLUNGEN:", bestellungen.columns.tolist())
print("KUNDEN:", kunden.columns.tolist())
print("PRODUKTE:", produkte.columns.tolist())

print("\nZeilenanzahl:")
print("Bestellungen:", len(bestellungen))
print("Kunden:", len(kunden))
print("Produkte:", len(produkte))

BESTELLUNGEN: ['bestell_id', 'datum', 'kunde_email', 'produkt', 'kategorie', 'menge', 'preis_eur', 'land', 'status', 'menge_pruefung', 'email_pruefung', 'umsatz_eur']
KUNDEN: ['kunde_id', 'name', 'ort', 'land', 'aktualisiert_am']
PRODUKTE: ['produkt_id', 'produkt_name', 'kategorie', 'preis_eur']

Zeilenanzahl:
Bestellungen: 210
Kunden: 31
Produkte: 12


In [2]:
print("KUNDENSTAMM:")
display(kunden.head(10))

print("PRODUKTKATALOG:")
display(produkte.head(10))

print("BESTELLUNGEN:")
display(bestellungen[["kunde_email", "produkt"]].head(10))

KUNDENSTAMM:


,kunde_id,name,ort,land,aktualisiert_am
0,K0001,Anna Berger,Köln,DE,2025-09-04
1,K0002,Jonas Wolf,Düsseldorf,DE,2025-10-08
2,K0003,Miriam Schuster,Nürnberg,DE,2025-08-12
3,K0004,Lukas Brandt,Osnabrück,DE,2025-02-28
4,K0005,Sofie Krämer,München,DE,2025-11-02
5,K0005,Sofie Krämer,Augsburg,DE,2026-03-14
6,K0006,David Lindner,Würzburg,DE,2025-12-25
7,K0007,Clara Vogt,Lübeck,DE,2025-01-05
8,K0008,Emil Hartmann,Erfurt,DE,2025-02-10
9,K0009,Lea Winkler,Freiburg,DE,2025-05-19


PRODUKTKATALOG:


,produkt_id,produkt_name,kategorie,preis_eur
0,SKU-101,Tastatur,Eingabegeräte,39.90
1,SKU-102,Maus,Eingabegeräte,19.90
2,SKU-103,Monitor 24 Zoll,Monitore & Zubehör,149.00
3,SKU-104,Headset,Audio & Video,59.90
4,SKU-105,Dockingstation,Kabel & Adapter,129.00
5,SKU-106,Laptopständer,Monitore & Zubehör,34.90
6,SKU-107,Webcam,Audio & Video,79.90
7,SKU-108,USB-C-Kabel,Kabel & Adapter,12.95
8,SKU-109,Mauspad,Eingabegeräte,9.95
9,SKU-110,Monitorarm,Monitore & Zubehör,89.00


BESTELLUNGEN:


,kunde_email,produkt
0,kontakt-fehlt,Dockingstation
1,hannes.vogel@example.com,Webcam HD
2,mia.lorenz@example.com,Beamer
3,tina.jaeger@example.com,Laptopständer
4,ida.neumann@example.com,Maus
5,NaN,Headset
6,nina.busch@example.com,Webcam HD
7,sophie.gross@example.com,USB-C-Kabel
8,erik.thiel@example.com,Webcam HD
9,olivia.krueger@example.com,Dockingstation


In [3]:
print("Alle Kundenspalten:")
print(kunden.columns.tolist())

print("\nAlle Bestellspalten:")
print(bestellungen.columns.tolist())

print("\nBeispielhafte Kunden:")
display(kunden[["kunde_id", "name"]].head(15))

print("\nE-Mail-Adressen aus Bestellungen:")
display(bestellungen["kunde_email"].dropna().drop_duplicates().head(15))

Alle Kundenspalten:
['kunde_id', 'name', 'ort', 'land', 'aktualisiert_am']

Alle Bestellspalten:
['bestell_id', 'datum', 'kunde_email', 'produkt', 'kategorie', 'menge', 'preis_eur', 'land', 'status', 'menge_pruefung', 'email_pruefung', 'umsatz_eur']

Beispielhafte Kunden:


,kunde_id,name
0,K0001,Anna Berger
1,K0002,Jonas Wolf
2,K0003,Miriam Schuster
3,K0004,Lukas Brandt
4,K0005,Sofie Krämer
5,K0005,Sofie Krämer
6,K0006,David Lindner
7,K0007,Clara Vogt
8,K0008,Emil Hartmann
9,K0009,Lea Winkler



E-Mail-Adressen aus Bestellungen:


0                  kontakt-fehlt
1       hannes.vogel@example.com
2         mia.lorenz@example.com
3        tina.jaeger@example.com
4        ida.neumann@example.com
6         nina.busch@example.com
7       sophie.gross@example.com
8         erik.thiel@example.com
9     olivia.krueger@example.com
10      lea.hartmann@example.com
11        tom.keller@example.com
12      clara.schmid@example.com
13       vera.sommer@example.com
15      ben.hoffmann@example.com
16     nils.schaefer@example.com
Name: kunde_email, dtype: str

In [4]:
kunden_namen = (
    kunden["name"]
    .str.lower()
    .str.replace(" ", ".", regex=False)
)

email_namen = (
    bestellungen["kunde_email"]
    .str.lower()
    .str.split("@").str[0]
)

print("Beispielhafte Namen aus dem Kundenstamm:")
print(kunden_namen.head(10).tolist())

print("\nE-Mail-Namen aus Bestellungen:")
print(email_namen.head(10).tolist())

print("\nÜbereinstimmende Namensmuster:")
print(email_namen.isin(kunden_namen).sum())

print("\nGesamtzahl Bestellpositionen:")
print(len(bestellungen))

Beispielhafte Namen aus dem Kundenstamm:
['anna.berger', 'jonas.wolf', 'miriam.schuster', 'lukas.brandt', 'sofie.krämer', 'sofie.krämer', 'david.lindner', 'clara.vogt', 'emil.hartmann', 'lea.winkler']

E-Mail-Namen aus Bestellungen:
['kontakt-fehlt', 'hannes.vogel', 'mia.lorenz', 'tina.jaeger', 'ida.neumann', nan, 'nina.busch', 'sophie.gross', 'erik.thiel', 'olivia.krueger']

Übereinstimmende Namensmuster:
6

Gesamtzahl Bestellpositionen:
210


## Join-Vertrag 1: Bestellungen und Kundenstamm

**Ziel:** Bestellungen mit Informationen aus dem Kundenstamm ergänzen.

**Join-Typ:** LEFT JOIN

**Begründung:** Alle 210 Bestellpositionen müssen erhalten bleiben.
Ein INNER JOIN könnte Bestellungen ohne passende Kundenzuordnung entfernen.

**Schlüsselproblem:**
- Bestellungen enthalten `kunde_email`.
- Der Kundenstamm enthält `kunde_id` und `name`, aber keine E-Mail-Adresse.
- Es existiert kein direkter gemeinsamer Kundenschlüssel.
- Eine Ableitung aus E-Mail-Namen ist nicht zuverlässig genug
  für eine bestätigte Kundenidentität.

**Umgang mit Dubletten:**
Mehrere Einträge mit derselben `kunde_id` werden anhand von
`aktualisiert_am` geprüft. Bei eindeutiger Aktualisierung wird
der neueste Datensatz bevorzugt.

**Qualitätsanforderungen:**
- Die Anzahl der Bestellpositionen darf sich nicht verändern.
- Der Umsatz darf durch den Join nicht künstlich steigen.
- Nicht zuordenbare Kunden müssen erkennbar bleiben.
- Es dürfen keine Kundenidentitäten erfunden werden.

**Offener Punkt:**
Für eine verlässliche Kundenverknüpfung wird eine eindeutige
Zuordnung zwischen Bestellungen und Kundenstamm benötigt.
Bis dahin sind Namensübereinstimmungen nur mögliche Kandidaten.

## Join-Vertrag 2: Bestellungen und Produktkatalog

**Ziel:** Bestellungen mit Produktinformationen aus dem Katalog ergänzen.

**Join-Typ:** LEFT JOIN

**Begründung:**
Alle 210 Bestellpositionen sollen erhalten bleiben.
Produkte ohne passende Katalogzuordnung dürfen nicht verschwinden.

**Join-Schlüssel:**
- Bestellungen: `produkt`
- Produktkatalog: `produkt_name`
- Beide Schlüssel werden vor dem Join vereinheitlicht.
- Leerzeichen und Groß-/Kleinschreibung werden normalisiert.
- Unterschiedliche Produktbezeichnungen werden nur nach
  nachvollziehbarer Prüfung einander zugeordnet.

**Erwartete Beziehung:** Viele Bestellungen zu einem Katalogprodukt (m:1).

**Validierung:**
Der Join wird mit `validate="m:1"` durchgeführt.
Dadurch verhindert pandas, dass mehrere Katalogeinträge
mit demselben Schlüssel unbemerkt Bestellzeilen vervielfachen.

**Qualitätsanforderungen:**
- Vorher: 210 Bestellpositionen.
- Nachher: ebenfalls 210 Bestellpositionen.
- Keine künstliche Umsatzsteigerung.
- Fehlende Produktzuordnungen werden gezählt.
- Produktpreise aus Bestellungen bleiben für die Umsatzberechnung maßgeblich.

**Phantom-Umsatz:**
Zusätzlicher Umsatz, der durch fehlerhafte Joins mit
mehrfach vorhandenen Schlüsseln entstehen könnte,
wird separat untersucht und beziffert.

In [7]:
# Doppelte Kundennummern untersuchen

kunden_dubletten = kunden[
    kunden.duplicated(subset=["kunde_id"], keep=False)
].sort_values(["kunde_id", "aktualisiert_am"])

print("Kundenstamm insgesamt:", len(kunden))
print("Unterschiedliche Kundennummern:", kunden["kunde_id"].nunique())
print("Zeilen mit mehrfach vorkommenden Kundennummern:", len(kunden_dubletten))

display(kunden_dubletten)

Kundenstamm insgesamt: 31
Unterschiedliche Kundennummern: 28
Zeilen mit mehrfach vorkommenden Kundennummern: 6


,kunde_id,name,ort,land,aktualisiert_am
4,K0005,Sofie Krämer,München,DE,2025-11-02
5,K0005,Sofie Krämer,Augsburg,DE,2026-03-14
12,K0012,Tim Rösler,Köln,DE,2025-08-21
13,K0012,Tim Rösler,Bonn,DE,2026-02-02
20,K0019,Ida Krüger,Wien,AT,2025-12-10
21,K0019,Ida Krüger,Graz,AT,2026-01-27


In [8]:
# Kundenstamm bereinigen: neueste Version je Kundennummer behalten

kunden_clean = kunden.copy()

# Datum korrekt umwandeln
kunden_clean["aktualisiert_am"] = pd.to_datetime(
    kunden_clean["aktualisiert_am"],
    errors="coerce"
)

# Neueste Version je Kunde behalten
kunden_clean = (
    kunden_clean
    .sort_values("aktualisiert_am")
    .drop_duplicates(subset=["kunde_id"], keep="last")
    .reset_index(drop=True)
)

# Qualitätsprüfungen
assert len(kunden_clean) == 28
assert kunden_clean["kunde_id"].is_unique
assert kunden_clean["aktualisiert_am"].notna().all()

print("Vorher:", len(kunden), "Kundeneinträge")
print("Nachher:", len(kunden_clean), "eindeutige Kunden")
print("Entfernte ältere Versionen:", len(kunden) - len(kunden_clean))

display(
    kunden_clean[
        kunden_clean["kunde_id"].isin(["K0005", "K0012", "K0019"])
    ]
)

Vorher: 31 Kundeneinträge
Nachher: 28 eindeutige Kunden
Entfernte ältere Versionen: 3


,kunde_id,name,ort,land,aktualisiert_am
25,K0019,Ida Krüger,Graz,AT,2026-01-27
26,K0012,Tim Rösler,Bonn,DE,2026-02-02
27,K0005,Sofie Krämer,Augsburg,DE,2026-03-14


In [9]:
# Mögliche Kunden-Zuordnungen untersuchen

kunden_check = kunden_clean.copy()

kunden_check["name_schluessel"] = (
    kunden_check["name"]
    .str.lower()
    .str.strip()
    .str.replace(" ", ".", regex=False)
)

print("Doppelte Namensschlüssel im bereinigten Kundenstamm:")
print(kunden_check["name_schluessel"].duplicated().sum())

print("\nMögliche Übereinstimmungen:")

kandidaten = bestellungen.copy()

kandidaten["email_schluessel"] = (
    kandidaten["kunde_email"]
    .str.lower()
    .str.split("@").str[0]
)

treffer = kandidaten[
    kandidaten["email_schluessel"].isin(kunden_check["name_schluessel"])
]

display(treffer[["bestell_id", "kunde_email", "email_schluessel"]])

Doppelte Namensschlüssel im bereinigten Kundenstamm:
0

Mögliche Übereinstimmungen:


,bestell_id,kunde_email,email_schluessel
48,3105,anna.berger@example.com,anna.berger
49,3039,anna.berger@example.com,anna.berger
79,3077,anna.berger@example.com,anna.berger
93,3086,anna.berger@example.com,anna.berger
108,3077,anna.berger@example.com,anna.berger
189,3206,anna.berger@example.com,anna.berger


In [10]:
# Kunden-Join: Namensmuster nur als unbestätigte Kandidaten

kunden_kandidaten = kandidaten.merge(
    kunden_check[
        ["kunde_id", "name", "name_schluessel"]
    ],
    how="left",
    left_on="email_schluessel",
    right_on="name_schluessel",
    validate="m:1",
    indicator=True
)

# Treffer ausdrücklich nicht als bestätigte Identität behandeln
kunden_kandidaten["zuordnung_status"] = (
    kunden_kandidaten["_merge"]
    .map({
        "both": "Kandidat – unbestätigt",
        "left_only": "Nicht zuordenbar"
    })
    .astype("string")
)

# Qualitätsprüfungen
assert len(kunden_kandidaten) == len(bestellungen)
assert kunden_kandidaten["umsatz_eur"].sum() == bestellungen["umsatz_eur"].sum()

print("Bestellpositionen vorher:", len(bestellungen))
print("Bestellpositionen nachher:", len(kunden_kandidaten))

print("\nZuordnungsstatus:")
print(kunden_kandidaten["zuordnung_status"].value_counts())

display(
    kunden_kandidaten[
        ["bestell_id", "kunde_email", "name", "zuordnung_status"]
    ].head(10)
)

Bestellpositionen vorher: 210
Bestellpositionen nachher: 210

Zuordnungsstatus:
zuordnung_status
Nicht zuordenbar          204
Kandidat – unbestätigt      6
Name: count, dtype: int64[pyarrow]


,bestell_id,kunde_email,name,zuordnung_status
0,3101,kontakt-fehlt,NaN,Nicht zuordenbar
1,3155,hannes.vogel@example.com,NaN,Nicht zuordenbar
2,3140,mia.lorenz@example.com,NaN,Nicht zuordenbar
3,3179,tina.jaeger@example.com,NaN,Nicht zuordenbar
4,3016,ida.neumann@example.com,NaN,Nicht zuordenbar
5,3171,NaN,NaN,Nicht zuordenbar
6,3074,nina.busch@example.com,NaN,Nicht zuordenbar
7,3208,sophie.gross@example.com,NaN,Nicht zuordenbar
8,3141,erik.thiel@example.com,NaN,Nicht zuordenbar
9,3190,olivia.krueger@example.com,NaN,Nicht zuordenbar


In [11]:
# Produktkatalog auf doppelte Schlüssel prüfen

print("Produkte insgesamt:", len(produkte))
print("Eindeutige Produkt-IDs:", produkte["produkt_id"].nunique())
print("Eindeutige Produktnamen:", produkte["produkt_name"].nunique())

print("\nDoppelte Produkt-IDs:")
display(
    produkte[
        produkte.duplicated("produkt_id", keep=False)
    ]
)

print("\nDoppelte Produktnamen:")
display(
    produkte[
        produkte.duplicated("produkt_name", keep=False)
    ]
)

Produkte insgesamt: 12
Eindeutige Produkt-IDs: 12
Eindeutige Produktnamen: 12

Doppelte Produkt-IDs:


,produkt_id,produkt_name,kategorie,preis_eur



Doppelte Produktnamen:


,produkt_id,produkt_name,kategorie,preis_eur


In [12]:
# Produktschlüssel normalisieren, ohne Originaldaten zu verändern

produkte_clean = produkte.copy()

produkte_clean["produkt_schluessel"] = (
    produkte_clean["produkt_name"]
    .str.strip()
    .str.lower()
)

bestellungen_produkte = kunden_kandidaten.copy()

bestellungen_produkte["produkt_schluessel"] = (
    bestellungen_produkte["produkt"]
    .str.strip()
    .str.lower()
)

print("Doppelte normalisierte Produktschlüssel:")
print(produkte_clean["produkt_schluessel"].duplicated().sum())

print("\nProdukte aus Bestellungen ohne Katalogtreffer:")

fehlende_produkte = (
    bestellungen_produkte.loc[
        ~bestellungen_produkte["produkt_schluessel"].isin(
            produkte_clean["produkt_schluessel"]
        ),
        "produkt"
    ]
    .value_counts()
)

display(fehlende_produkte)

Doppelte normalisierte Produktschlüssel:
0

Produkte aus Bestellungen ohne Katalogtreffer:


produkt
Webcam HD          19
Monitor 27 Zoll    19
Lautsprecher       17
Beamer              3
Name: count, dtype: int64

In [13]:
# Gesamten Produktkatalog anzeigen

display(
    produkte_clean[
        ["produkt_id", "produkt_name", "kategorie", "preis_eur"]
    ]
)

,produkt_id,produkt_name,kategorie,preis_eur
0,SKU-101,Tastatur,Eingabegeräte,39.90
1,SKU-102,Maus,Eingabegeräte,19.90
2,SKU-103,Monitor 24 Zoll,Monitore & Zubehör,149.00
3,SKU-104,Headset,Audio & Video,59.90
4,SKU-105,Dockingstation,Kabel & Adapter,129.00
5,SKU-106,Laptopständer,Monitore & Zubehör,34.90
6,SKU-107,Webcam,Audio & Video,79.90
7,SKU-108,USB-C-Kabel,Kabel & Adapter,12.95
8,SKU-109,Mauspad,Eingabegeräte,9.95
9,SKU-110,Monitorarm,Monitore & Zubehör,89.00


In [14]:
# Produktkatalog sicher mit Bestellungen verknüpfen

daten_gesamt = bestellungen_produkte.merge(
    produkte_clean[
        ["produkt_schluessel", "produkt_id", "produkt_name", "kategorie"]
    ],
    how="left",
    on="produkt_schluessel",
    validate="m:1",
    indicator="produkt_join_status"
)

# Qualitätsprüfungen
assert len(daten_gesamt) == len(bestellungen)
assert daten_gesamt["umsatz_eur"].sum() == bestellungen["umsatz_eur"].sum()

print("Bestellpositionen vorher:", len(bestellungen))
print("Bestellpositionen nachher:", len(daten_gesamt))

print("\nProduktzuordnungen:")
print(daten_gesamt["produkt_join_status"].value_counts())

print("\nUmsatz vor Join:", round(bestellungen["umsatz_eur"].sum(), 2))
print("Umsatz nach Join:", round(daten_gesamt["umsatz_eur"].sum(), 2))

display(
    daten_gesamt[
        ["bestell_id", "produkt", "produkt_name", "produkt_id"]
    ].head(10)
)

Bestellpositionen vorher: 210
Bestellpositionen nachher: 210

Produktzuordnungen:
produkt_join_status
both          152
left_only      58
right_only      0
Name: count, dtype: int64

Umsatz vor Join: 41076.7
Umsatz nach Join: 41076.7


,bestell_id,produkt,produkt_name,produkt_id
0,3101,Dockingstation,Dockingstation,SKU-105
1,3155,Webcam HD,NaN,NaN
2,3140,Beamer,NaN,NaN
3,3179,Laptopständer,Laptopständer,SKU-106
4,3016,Maus,Maus,SKU-102
5,3171,Headset,Headset,SKU-104
6,3074,Webcam HD,NaN,NaN
7,3208,USB-C-Kabel,USB-C-Kabel,SKU-108
8,3141,Webcam HD,NaN,NaN
9,3190,Dockingstation,Dockingstation,SKU-105


In [15]:
# Umsatzbericht: Nur abgeschlossene Bestellungen

bericht = daten_gesamt[
    daten_gesamt["status"] == "abgeschlossen"
].copy()

gesamtumsatz = bericht["umsatz_eur"].sum()

umsatz_kategorie = (
    bericht.groupby("kategorie_x", dropna=False)["umsatz_eur"]
    .sum()
    .sort_values(ascending=False)
)

umsatz_land = (
    bericht.groupby("land", dropna=False)["umsatz_eur"]
    .sum()
    .sort_values(ascending=False)
)

print(f"Gesamtumsatz Q1: {gesamtumsatz:,.2f} €")

print("\nUmsatz nach Kategorie:")
display(umsatz_kategorie)

print("\nUmsatz nach Land:")
display(umsatz_land)

# Kontrollieren, ob die Summen übereinstimmen
assert abs(umsatz_kategorie.sum() - gesamtumsatz) < 0.01
assert abs(umsatz_land.sum() - gesamtumsatz) < 0.01

Gesamtumsatz Q1: 37,263.80 €

Umsatz nach Kategorie:


kategorie_x
Zubehör     13852.4
Monitore     9960.0
Video        7136.0
Audio        6315.4
Name: umsatz_eur, dtype: Float64


Umsatz nach Land:


land
DE    29601.1
AT     6232.8
CH     1429.9
Name: umsatz_eur, dtype: Float64

In [16]:
# Phantom-Umsatz durch doppelte Kundenschlüssel demonstrieren

# Ausgangspunkt: eindeutiger Kundenstamm
kunden_basis = kunden_clean[["kunde_id", "name"]].copy()

# Simulation: drei Kundennummern absichtlich verdoppeln
kunden_doppelt = pd.concat(
    [
        kunden_basis,
        kunden_basis[
            kunden_basis["kunde_id"].isin(
                ["K0005", "K0012", "K0019"]
            )
        ]
    ],
    ignore_index=True
)

print("Kundenstamm ohne Dubletten:", len(kunden_basis))
print("Simulierter Kundenstamm mit Dubletten:", len(kunden_doppelt))

print("\nDoppelte Kundennummern:")
display(
    kunden_doppelt[
        kunden_doppelt.duplicated("kunde_id", keep=False)
    ].sort_values("kunde_id")
)

Kundenstamm ohne Dubletten: 28
Simulierter Kundenstamm mit Dubletten: 31

Doppelte Kundennummern:


,kunde_id,name
27,K0005,Sofie Krämer
30,K0005,Sofie Krämer
26,K0012,Tim Rösler
29,K0012,Tim Rösler
25,K0019,Ida Krüger
28,K0019,Ida Krüger


In [17]:
# Phantom-Umsatz bei einem fehlerhaften Join simulieren

# Nur abgeschlossene Bestellungen berücksichtigen
simulation_basis = bericht.copy()

# Absichtlich einen doppelten Namensschlüssel erzeugen
kunden_simulation = pd.concat(
    [
        kunden_check[["name_schluessel", "kunde_id"]],
        kunden_check.loc[
            kunden_check["name_schluessel"] == "anna.berger",
            ["name_schluessel", "kunde_id"]
        ]
    ],
    ignore_index=True
)

# Fehlerhaften Join simulieren
simulation_join = simulation_basis.merge(
    kunden_simulation,
    how="left",
    left_on="email_schluessel",
    right_on="name_schluessel"
)

umsatz_original = simulation_basis["umsatz_eur"].sum()
umsatz_simulation = simulation_join["umsatz_eur"].sum()
phantom_umsatz = umsatz_simulation - umsatz_original

print("Bestellpositionen vorher:", len(simulation_basis))
print("Bestellpositionen nach fehlerhaftem Join:", len(simulation_join))

print(f"\nUmsatz vorher: {umsatz_original:,.2f} €")
print(f"Umsatz nach fehlerhaftem Join: {umsatz_simulation:,.2f} €")
print(f"Phantom-Umsatz: {phantom_umsatz:,.2f} €")

Bestellpositionen vorher: 191
Bestellpositionen nach fehlerhaftem Join: 197

Umsatz vorher: 37,263.80 €
Umsatz nach fehlerhaftem Join: 38,547.25 €
Phantom-Umsatz: 1,283.45 €


In [18]:
# Nachweisen, dass validate="m:1" doppelte Schlüssel erkennt

try:
    simulation_basis.merge(
        kunden_simulation,
        how="left",
        left_on="email_schluessel",
        right_on="name_schluessel",
        validate="m:1"
    )

    print("WARNUNG: Der Join wurde unerwartet zugelassen!")

except pd.errors.MergeError:
    print("Sicherheitsprüfung erfolgreich!")
    print("Pandas hat den Join wegen doppelter Kundenschlüssel verhindert.")
    print("Dadurch entsteht kein zusätzlicher Phantom-Umsatz.")

Sicherheitsprüfung erfolgreich!
Pandas hat den Join wegen doppelter Kundenschlüssel verhindert.
Dadurch entsteht kein zusätzlicher Phantom-Umsatz.


## Ergebnisse und Bewertung der Joins

### Kundenstamm
Der Kundenstamm enthielt ursprünglich 31 Einträge mit 28 eindeutigen Kundennummern.

Bei drei Kundennummern waren unterschiedliche Versionen vorhanden. Für die Bereinigung wurde jeweils der zuletzt aktualisierte Datensatz übernommen. Die ursprünglichen Daten blieben unverändert.

Da die Bestellungen keine verlässliche Kundennummer enthalten, war eine sichere Verknüpfung nicht möglich. Sechs Bestellpositionen konnten anhand von Namensmustern als mögliche Treffer identifiziert werden. Diese wurden ausdrücklich als unbestätigte Kandidaten gekennzeichnet.

### Produktkatalog
Der Produktkatalog enthält 12 eindeutige Produkte.

Nach der Normalisierung der Produktnamen konnten 152 von 210 Bestellpositionen zugeordnet werden.

58 Positionen blieben ohne Katalogtreffer. Betroffen waren Webcam HD, Monitor 27 Zoll, Lautsprecher und Beamer.

Diese Produkte wurden nicht automatisch anderen Katalogartikeln zugeordnet, da dadurch falsche Produktinformationen entstehen könnten.

### Umsatzbericht
Der Umsatz aus abgeschlossenen Bestellungen im ersten Quartal 2026 beträgt 37.263,80 €.

Die Summen nach Kategorie und Land stimmen mit dem Gesamtumsatz überein.

Durch die regulären Joins entstanden weder zusätzliche Bestellpositionen noch künstliche Umsätze.

### Phantom-Umsatz
In einer Simulation wurde ein Kundenschlüssel absichtlich verdoppelt.

Dadurch stieg die Anzahl abgeschlossener Bestellpositionen von 191 auf 197.

Der berechnete Umsatz erhöhte sich von 37.263,80 € auf 38.547,25 €.

Der simulierte Phantom-Umsatz betrug somit 1.283,45 € beziehungsweise rund 3,44 %.

Mit `validate="m:1"` konnte dieser fehlerhafte Join erfolgreich verhindert werden.

### Fazit
Ein technisch ausführbarer Join ist nicht automatisch fachlich korrekt.

Vor jeder Tabellenverknüpfung müssen die Schlüssel, ihre Eindeutigkeit und die erwartete Beziehung geprüft werden.

Fehlende Zuordnungen sollten dokumentiert werden, statt durch unbegründete Annahmen falsche Daten zu erzeugen.

## Copilot-Reflexion: KI-Vorschläge kritisch prüfen

### 1. INNER JOIN statt LEFT JOIN

Ein KI-Assistent könnte einen `inner`-Join vorschlagen, um nur Datensätze mit passenden Schlüsseln zusammenzuführen.

Für unsere Aufgabe wäre das problematisch:

- Beim Produkt-Join würden 58 Bestellpositionen ohne Katalogtreffer verschwinden.
- Dadurch wäre der Umsatzbericht unvollständig.
- Fehlende Produktzuordnungen wären anschließend nicht mehr sichtbar.

**Unsere Entscheidung:** Wir verwenden `how="left"`, damit sämtliche Bestellpositionen erhalten bleiben.

### 2. Fehlende Werte mit astype(str) umwandeln

Ein KI-Assistent könnte empfehlen, die E-Mail-Spalte mit `astype(str)` umzuwandeln.

Dabei könnten fehlende Werte (`NaN`) in die Zeichenfolge `"nan"` umgewandelt werden.

Das wäre problematisch, weil ein fehlender Wert dadurch wie ein tatsächlich vorhandener Text behandelt werden könnte.

**Unsere Entscheidung:** Wir verwenden Pandas-Stringmethoden, die fehlende Werte erhalten, und erfinden keine Kundenidentitäten.

### 3. KI-Ergebnisse müssen überprüft werden

Die Simulation hat gezeigt, dass ein fehlerhafter Join einen Phantom-Umsatz von 1.283,45 € erzeugen könnte.

Mit `validate="m:1"` konnten wir diesen Fehler verhindern.

**Fazit:** KI kann bei der Erstellung von Code unterstützen. Die Verantwortung für Datenqualität, fachliche Entscheidungen und die Prüfung der Ergebnisse bleibt jedoch beim Menschen.

In [19]:
# Abschlussprüfung – Woche 2, Tag 2

# 1. Keine Bestellpositionen verloren
assert len(daten_gesamt) == len(bestellungen) == 210

# 2. Kundenschlüssel eindeutig
assert kunden_clean["kunde_id"].is_unique

# 3. Produktschlüssel eindeutig
assert produkte_clean["produkt_schluessel"].is_unique

# 4. Umsatz durch Joins unverändert
assert abs(
    daten_gesamt["umsatz_eur"].sum()
    - bestellungen["umsatz_eur"].sum()
) < 0.01

# 5. Q1-Umsatz korrekt
assert abs(gesamtumsatz - 37263.80) < 0.01

# 6. Produktzuordnungen korrekt gezählt
assert (daten_gesamt["produkt_join_status"] == "both").sum() == 152
assert (daten_gesamt["produkt_join_status"] == "left_only").sum() == 58

print("✅ Alle Qualitätsprüfungen bestanden!")
print("✅ 210 Bestellpositionen erhalten")
print("✅ Kundenschlüssel eindeutig")
print("✅ Produktschlüssel eindeutig")
print("✅ Keine künstliche Umsatzsteigerung")
print("✅ Q1-Umsatz: 37.263,80 €")
print("✅ 152 Produktzuordnungen, 58 fehlende")
print("✅ Simulierter Phantom-Umsatz: 1.283,45 €")

✅ Alle Qualitätsprüfungen bestanden!
✅ 210 Bestellpositionen erhalten
✅ Kundenschlüssel eindeutig
✅ Produktschlüssel eindeutig
✅ Keine künstliche Umsatzsteigerung
✅ Q1-Umsatz: 37.263,80 €
✅ 152 Produktzuordnungen, 58 fehlende
✅ Simulierter Phantom-Umsatz: 1.283,45 €
